# Lily 1.5b v0.3 Non-GGUF Model Inference — Google Colab
**Interactive Reasoning & Schema Auditing for `abhinav0231/Lily-1.5b-v0.3`**

This notebook loads the un-quantized 16-bit standalone model `abhinav0231/Lily-1.5b-v0.3` from Hugging Face, executes ChatML reasoning inference, parses `<think>` and `<answer>` tags, and performs programmatic schema compliance auditing.

## Cell 1 — Install Transformers & Dependencies

In [ ]:
# ==============================================================================
# Cell 1 — Install Hugging Face Transformers & Acceleration Libraries
# ==============================================================================
!pip install -q -U transformers accelerate sentencepiece

## Cell 2 — Load Non-GGUF Model (`abhinav0231/Lily-1.5b-v0.3`)

In [ ]:
# ==============================================================================
# Cell 2 — Load Hugging Face Model & Tokenizer
# ==============================================================================
import re
import json
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

# Target un-quantized 16-bit distilled model repository on Hugging Face Hub
repo_id = "abhinav0231/Lily-1.5b-v0.3"

tokenizer = AutoTokenizer.from_pretrained(repo_id)

model = AutoModelForCausalLM.from_pretrained(
    repo_id,
    torch_dtype=torch.float16,
    device_map="auto",
)

print("✅ model + tokenizer loaded")
print("device:", model.device)

## Cell 3 — Inference Function & Tag Parser

In [ ]:
# ==============================================================================
# Cell 3 — Prompt Generation & CoT Regex Parsing
# ==============================================================================
SYSTEM_PROMPT = (
    "You are a precise, helpful assistant. "
    "Always reason step by step inside <think></think> tags, "
    "then write your final answer inside <answer></answer> tags."
)

def ask(question, max_new_tokens=1024, temperature=0.7):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": question},
    ]
    input_ids = tokenizer.apply_chat_template(
        messages,
        tokenize              = True,
        add_generation_prompt = True,
        return_tensors        = "pt",
    ).to(model.device)

    output_ids = model.generate(
        input_ids,
        max_new_tokens = max_new_tokens,
        temperature    = temperature,
        top_p          = 0.95,
        do_sample      = True if temperature > 0 else False,
        pad_token_id   = tokenizer.eos_token_id,
    )

    response = tokenizer.decode(output_ids[0][input_ids.shape[-1]:], skip_special_tokens=True)
    return response

def parse_and_print(response):
    think_m  = re.search(r"<think>(.*?)</think>", response, re.DOTALL)
    answer_m = re.search(r"<answer>(.*?)</answer>", response, re.DOTALL)
    
    print("🧠 REASONING (<think>):")
    print(think_m.group(1).strip() if think_m else "No <think> tag found.")
    print("\n🎯 FINAL ANSWER (<answer>):")
    print(answer_m.group(1).strip() if answer_m else response.split("</think>")[-1].strip())

print("✅ Functions ready")

## Cell 4 — Programmatic Schema Compliance Auditor

In [ ]:
# ==============================================================================
# Cell 4 — Programmatic Schema Compliance Engine
# Audits tag health (<think>, </think>, <answer>, </answer>, and <thinking> leaks)
# ==============================================================================
def extract_stats(text):
    return {
        "has_think_open":   "<think>" in text,
        "has_think_close":  "</think>" in text,
        "has_answer_open":  "<answer>" in text,
        "has_answer_close": "</answer>" in text,
        "think_count":     text.count("<think>"),
        "thinking_count":  text.count("<thinking>"), # Audit tag leakage
        "answer_count":    text.count("<answer>"),
        "im_end_count":    text.count("<|im_end|>"),
    }

def classify_case(raw_stats):
    if raw_stats["thinking_count"] > 0:
        return "FAIL_thinking_leak"
    if raw_stats["think_count"] > 1:
        return "FAIL_duplicate_think"
    if (
        raw_stats["has_think_open"]
        and raw_stats["has_think_close"]
        and raw_stats["has_answer_open"]
        and raw_stats["has_answer_close"]
    ):
        return "PASS_exact"
    if raw_stats["has_think_open"] and raw_stats["has_think_close"]:
        return "PARTIAL_only_think"
    return "FAIL_no_schema"

print("✅ Schema Compliance Auditor defined")